In [ ]:
import sys, numpy as np, pandas as pd, matplotlib
print(sys.version)
print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)


In [2]:
from model.params import load_params, summarize
mp = load_params("params.yaml")          # or load_params("params.yaml", scenario="easy_migration")
print(summarize(mp))


=== Globals ===
alpha=0.33, delta=0.06, phi_M=0.9, g_A=0.0, q_star=1.0, eta=0.3
== D == s=0.26, f=0.003, eps=0.45, kappa=0.2, xi=0.45, theta_bar=0.85, zeta=0.92
== L == s=0.2, f=0.015, eps=0.7, kappa=0.2, xi=0.25, theta_bar=0.65, zeta=0.98
== Migration == mu=0.06, tau_H=0.08, m_bar=0.1, epsilon_slack=0.02
== Initial == K={'D': 1.0, 'L': 1.0}, N={'D': 1.0, 'L': 1.0}, A0={'D': 1.0, 'L': 1.0}, M0=0.0
== Simulation == T=200


In [3]:
from model.params import load_params
from model.solow import steady_state_country

mp = load_params("params.yaml")
resD = steady_state_country(mp.D, mp.globals, mp.toggles, n=mp.D.f)
resL = steady_state_country(mp.L, mp.globals, mp.toggles, n=mp.L.f)

print("D steady state:", resD)
print("L steady state:", resL)


AttributeError: 'ModelParams' object has no attribute 'toggles'

In [ ]:
from model.params import load_params
from model.simulate import simulate, empirical_growth_rate

mp = load_params("params.yaml")
res = simulate(mp, T=50)

print("Final states:")
print("A_T =", res.A[-1], " M_T =", res.M[-1])
print("N_D_T =", res.D.N[-1], " N_L_T =", res.L.N[-1])
print("K_D_T =", res.D.K[-1], " K_L_T =", res.L.K[-1])

print("\nSample ratios (M_t/E_t):", res.M_over_E[-5:])
print("Approx g_E (log average, last 10):", empirical_growth_rate(res.E_world, window=10))


In [ ]:
from experiments.runner import run_all, write_summary_csv

res = run_all(base_yaml_path="params.yaml", T=100)
write_summary_csv(res["records"], out_path="results/experiments_summary.csv")

len(res["records"]), list(res["raw"].keys()), list(res["raw"]["off"].keys())


In [ ]:
from experiments.runner import run_all
from viz.plots import capital_dilution_plot, emissions_and_ratio_plot, ambiguity_map
import numpy as np

res_all = run_all(base_yaml_path="params.yaml", T=80)
rec = res_all["raw"]["off"]["nD_up"]

capital_dilution_plot(rec["res_base"], rec["res_shock"], "nD_up", "off")
emissions_and_ratio_plot(rec["res_base"], rec["res_shock"], "nD_up", "off")
ambiguity_map(np.linspace(0.1,1.0,50), np.linspace(0.1,3.0,50), eta=0.5)


In [ ]:
from experiments.runner import run_all
from viz.tables import save_tables

res_all = run_all(base_yaml_path="params.yaml", T=120)
dfs = save_tables(res_all, outdir="results")
dfs["combined"].head()



In [ ]:
from model.params import load_params
from model.simulate import simulate
from model.checks import check_simulation, assert_simulation_ok

mp = load_params("params.yaml")
res = simulate(mp, T=80)

report = check_simulation(res, mp)
print("OK?", report.ok)
for m in report.messages:
    print("-", m)

# This one raises an error if anything is wrong:
assert_simulation_ok(res, mp)


In [ ]:
from model.params import load_params
mp = load_params("params.yaml")

print("Active labor share (ϛ):", getattr(mp.D, "varsigma", None), getattr(mp.L, "varsigma", None))
print("Eta for income-dependent intensity:", mp.globals.eta)

